# Group G
dataset: https://www.kaggle.com/datasets/sobhanmoosavi/us-weather-events/data

In [ ]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
# This might be needed in order to get plots to display in the exported HTML for submission.
# In any case, please double check that plots display properly before you submit.
import plotly.io as pio
pio.renderers.default='notebook'
df = pd.read_csv("WeatherEvents_Jan2016-Dec2022.csv")

In [ ]:
fig = px.histogram(df, y=df.Type, title="Conteggio totale per tipo di evento")
fig.update_layout(yaxis={'categoryorder': 'total ascending'})
fig.show()

In [ ]:
fig = px.scatter(df, y="LocationLat", x="LocationLng")
fig.update_layout(hovermode=False)
fig.show()

In [ ]:
#Selezionare eventi atmosferici storici che sono avvenuti nell'arco del tempo del nostro dataset e mostrarli con dei grafici (magari lo spostamento di un huragano (grafico animato))

#Grafico HeatMap che mostra l'andamento degli eventi nei vari mesi

## Distribuzione Oraria degli Eventi Meteo

In [ ]:
df['StartTime(UTC)'] = pd.to_datetime(df['StartTime(UTC)'], utc=True)
df['EndTime(UTC)'] = pd.to_datetime(df['EndTime(UTC)'], utc=True)

def to_local(group):
    tz_name = group.name 
    group['Local_Time'] = group['StartTime(UTC)'].dt.tz_convert(tz_name).dt.tz_localize(None)
    return group

hourly_counts = df.groupby('TimeZone', group_keys=False).apply(to_local)

hourly_counts['Local_Hour'] = hourly_counts['Local_Time'].dt.hour

In [ ]:
hourly_counts = hourly_counts.groupby(['Local_Hour', 'Type']).size().reset_index(name='Event_Count')

# Creiamo il grafico
fig = px.line(
    hourly_counts, 
    x = 'Local_Hour', 
    y = 'Event_Count', 
    color = 'Type',
    title = 'Distribuzione Oraria degli Eventi Meteo (Ora Locale)',
    labels = {'Local_Hour': 'Ora del Giorno (0-23)', 'Event_Count': 'Numero di Eventi'},
    template = 'plotly_white'
)

fig.update_layout(xaxis=dict(tickmode='linear', tick0=0, dtick=1))

fig.show()

## Percentuale Oraria degli Eventi Meteo

In [ ]:
hourly_counts['Percentage'] = hourly_counts.groupby('Type')['Event_Count'].transform(
    lambda x: (x / x.sum()) * 100
)

# Creiamo il grafico
fig = px.line(
    hourly_counts, 
    x = 'Local_Hour', 
    y = 'Percentage', 
    color = 'Type',
    title = 'Percentuale Oraria degli Eventi Meteo (Ora Locale)',
    labels = {'Local_Hour': 'Ora del Giorno (0-23)', 'Event_Count': 'Numero di Eventi'},
    template = 'plotly_white'
)

fig.update_layout(xaxis=dict(tickmode='linear', tick0=0, dtick=1))

fig.show()

## Precipitazioni Medie Annuali

In [ ]:
df["Year"] = df["StartTime(UTC)"].dt.year
df["Precipitation(mm)"] = df["Precipitation(in)"] * 25.4

mask = df["Type"] == "Rain"
annual_per_station = df.groupby(["Year", "AirportCode"])["Precipitation(mm)"].sum().reset_index()

final_yearly_data = annual_per_station.groupby("Year")["Precipitation(mm)"].mean().reset_index()


fig = px.bar(
    final_yearly_data,
    x = "Year",
    y = "Precipitation(mm)",
    title="Precipitazioni Medie Annuali per Stazione negli USA",
    labels={
        "Precipitation(mm)": "Media Pioggia Annua [mm]", 
        "Year": "Anno"
    },
    template = "plotly_white"
    )

# non mi convince questo grafico
fig.show()

## Distribuzione annuale per Neve e Nebbia

In [ ]:
snow_year = df[df["Type"] == "Snow"].groupby("Year").size().reset_index(name = "SnowCount")
fog_year = df[df["Type"] == "Fog"].groupby("Year").size().reset_index(name = "FogCount")

fig = make_subplots(
    rows=2, cols=1, 
    shared_xaxes=True, 
    vertical_spacing=0.1,
    subplot_titles=("Distribuzione Annuale Neve", "Distribuzione Annuale Nebbia")
)


fig.add_trace(
    go.Bar(
        x = snow_year["Year"],
        y = snow_year["SnowCount"], 
        name = "Neve",
        marker_color = "skyblue"
    ),
    row=1, col=1
)


fig.add_trace(
    go.Bar(
        x = fog_year["Year"],
        y = fog_year["FogCount"], 
        name = "Nebbia",
        marker_color = "lightgrey"
    ),
    row=2, col=1
)

fig.update_layout(
    title_text = "Analisi Eventi Meteo per Anno",
    template = "plotly_white",
    showlegend = False,
    height = 700
)

fig.update_yaxes(title_text="Conteggio Neve", row=1, col=1)
fig.update_yaxes(title_text="Conteggio Nebbia", row=2, col=1)
fig.update_xaxes(title_text="Anno", row=2, col=1)

fig.show()

## Intensità piogga negli anni
(lo toglieri)

In [ ]:

# 1. Preparazione dati (identica a prima)
rain = df[df["Type"] == "Rain"].copy()
rain["DurationHrs"] = (rain["EndTime(UTC)"] - rain["StartTime(UTC)"]).dt.total_seconds() / 3600

rain = rain[
    (rain["DurationHrs"] > 0) & 
    (rain["Precipitation(mm)"] > 0) & 
    rain["LocationLat"].notna() & 
    rain["LocationLng"].notna()
]

rain["Intensity"] = rain["Precipitation(mm)"] / rain["DurationHrs"]

agg = rain.groupby(
    ["Year", "AirportCode", "City", "State", "LocationLat", "LocationLng"]
).agg(
    EventCount=("EventId", "count"),
    AvgIntensity=("Intensity", "mean"),
    AvgDurationHr=("DurationHrs", "mean"),
    TotalPrecipIn=("Precipitation(mm)", "sum")
).reset_index().sort_values("Year")

p99 = agg["AvgIntensity"].quantile(0.99)
agg["IntensityClipped"] = agg["AvgIntensity"].clip(upper=p99)

# 2. Creazione mappa con Plotly Express
fig = px.scatter_geo(
    agg,
    lat="LocationLat",
    lon="LocationLng",
    color="IntensityClipped",
    size="EventCount",
    animation_frame="Year",
    hover_name="AirportCode",
    hover_data={
        "City": True, 
        "State": True,
        "LocationLat": False, 
        "LocationLng": False,
        "IntensityClipped": False,
        "AvgIntensity": ":.4f",
        "AvgDurationHr": ":.2f",
        "TotalPrecipIn": ":.2f"
    },
    color_continuous_scale="Blues",
    range_color=[0, p99],
    size_max=8,
    scope="usa",
    title="Intensità pioggia negli USA per stazione meteo",
    labels={"IntensityClipped": "Intensità (mm/hr)"}
)

# 3. Aggiornamento estetico (colori scuri come nel tuo esempio)
fig.update_layout(
    geo=dict(
        projection_type="albers usa",
        showland=True, landcolor="#1a1a2e",
        showlakes=True, lakecolor="#16213e",
        showcoastlines=True, coastlinecolor="#444",
        showsubunits=True, subunitcolor="#333",
        bgcolor="#0f0f1a"
    ),
    paper_bgcolor="#0f0f1a",
    plot_bgcolor="#0f0f1a",
    font=dict(color="white")
)

fig.show()

## Intensità pioggia per anni per stato
(toglierei anche questo frse)

In [ ]:

rain = df[df["Type"] == "Rain"].copy()

rain["DurationHrs"] = (rain["EndTime(UTC)"] - rain["StartTime(UTC)"]).dt.total_seconds() / 3600

rain = rain[
    (rain["DurationHrs"] > 0) & 
    (rain["Precipitation(mm)"] > 0) & 
    rain["State"].notna()
]

rain["Intensity"] = rain["Precipitation(mm)"] / rain["DurationHrs"]

agg = rain.groupby(["Year", "State"]).agg(
    EventCount=("EventId", "count"),
    AvgIntensity=("Intensity", "mean"),
    AvgDurationHr=("DurationHrs", "mean"),
    TotalPrecipIn=("Precipitation(in)", "sum")
).reset_index().sort_values("Year")

p99 = agg["AvgIntensity"].quantile(0.99)
agg["IntensityClipped"] = agg["AvgIntensity"].clip(upper=p99)

fig = px.choropleth(
    agg,
    locations="State",
    locationmode="USA-states",
    color="IntensityClipped",
    animation_frame="Year",
    hover_name="State",
    hover_data={
        "State": False,
        "IntensityClipped": False,
        "AvgIntensity": ":.4f",
        "AvgDurationHr": ":.2f",
        "TotalPrecipIn": ":.2f",
        "EventCount": True
    },
    color_continuous_scale="Blues",
    range_color=[0, p99],
    scope="usa",
    title="Intensità media pioggia negli USA per stato",
    labels={"IntensityClipped": "Intensità (mm/hr)"}
)

fig.update_traces(
    marker_line_width=0.5, 
    marker_line_color="#ffffff"
)

fig.update_layout(
    geo=dict(
        bgcolor="#0f0f1a",
        lakecolor="#16213e",
        showlakes=True,
        showcoastlines=False,
        subunitcolor="#555"
    ),
    paper_bgcolor="#0f0f1a",
    plot_bgcolor="#0f0f1a",
    font=dict(color="white"),
    margin=dict(l=0, r=0, t=60, b=90),
    height=620
)

fig.show()

## Movimento di Harvey dal 26 al 29 agosto
da modificaee qulcosa, così non è molto esplicativo

In [ ]:
hours = pd.date_range("2017-08-26", "2017-08-29 23:00", freq="h", tz="UTC")

stations = (
    df[df["StartTime(UTC)"].dt.year == 2017]
    .dropna(subset=["LocationLat", "LocationLng"])
    .drop_duplicates("AirportCode")[["LocationLat", "LocationLng"]]
)

rain = df[
    (df["Type"] == "Rain") &
    (df["Severity"].isin(["Heavy"])) &
    (df["StartTime(UTC)"] <= hours[-1]) &
    (df["EndTime(UTC)"] >= hours[0])
].dropna(subset=["LocationLat", "LocationLng"]).copy()

active_frames = []
for h in hours:
    active = rain[(rain["StartTime(UTC)"] <= h) & (rain["EndTime(UTC)"] > h)].copy()
    if not active.empty:
        active["CurrentHour"] = h.strftime("%Y-%m-%d %H:%M UTC")
        active_frames.append(active)

df_anim = pd.concat(active_frames, ignore_index=True)
df_anim["Severity"] = pd.Categorical(df_anim["Severity"], categories=["Heavy"], ordered=True)
df_anim = df_anim.sort_values(["CurrentHour", "Severity"])

fig = px.scatter_geo(
    df_anim,
    lat="LocationLat",
    lon="LocationLng",
    color="Severity",
    animation_frame="CurrentHour",
    hover_name="City",
    hover_data={"State": True, "Severity": False, "LocationLat": False, "LocationLng": False, "CurrentHour": False},
    #color_discrete_map={"Heavy": "#42a5f5", "Severe": "#ff6f00"},
    scope="usa",
    title="Harvey — Pioggia intensa (27–29 Agosto 2017)"
)

fig.add_trace(go.Scattergeo(
    lat=stations["LocationLat"], lon=stations["LocationLng"], mode="markers",
    marker=dict(size=4, color="#37474f", opacity=0.2), hoverinfo="skip", showlegend=False
))

fig.update_traces(
    marker=dict(size=8, opacity=0.9, line=dict(width=0.5, color="white")),
    selector=dict(type="scattergeo", showlegend=True)
)

fig.update_layout(
    geo=dict(
        projection_type="albers usa", showland=True, landcolor="#1a1a2e",
        showcoastlines=True, coastlinecolor="#37474f", showsubunits=True, subunitcolor="#2e3e4e",
        bgcolor="#0f0f1a"
    ),
    paper_bgcolor="#0f0f1a", plot_bgcolor="#0f0f1a", font=dict(color="white"),
    margin=dict(l=0, r=0, t=50, b=80)
)

fig.show()

## Heatmap della severità per evento

In [ ]:
mask = (df['Severity'] == 'Severe') & (df["Type"] == "Fog")
df_severe_fog = df[mask].copy()
#il copy va usato perche pandas ritorna una vista del dataseto originale se dopo modifichiamo df_severe potrebbe dare errore

df_severe_fog['Month'] = df_severe_fog['StartTime(UTC)'].dt.month
#trasformandolo in datetime di pandas ci possiamo accedere con dt e abbiamo le sue funzionalità

#Creiamo la heatmap

#Contiamo per ongi combinazione di mese-anno il numero di eventi severe (collassa tutte le righe Gennaio 2016 in una sola contando il numero di occorrenze)
heatmap = df_severe_fog.groupby(['Year','Month']).size().reset_index(name='Count')
#Quindi dopo qui abbiamo le righe singole che sono Gennaio 2016 / febbraio 2016 ... agosto 2016 e per ognuna di queste righe uniche abbiamo il numero di count 

#Creiamo una tabella pivot da passare alla heatmap
pivot_table = heatmap.pivot(index='Year', columns='Month', values='Count').fillna(0)

fig = px.imshow(pivot_table, 
                title="Stagionalità e Frequenza degli Eventi Estremi di Nebbia (2016-2022)",
                labels=dict(x="Mese", y="Anno", color="N eventi Estremi"),
                x=['Gen', 'Feb', 'Mar', 'Apr', 'Mag', 'Giu', 'Lug', 'Ago', 'Set', 'Ott', 'Nov', 'Dic']
                );

fig.show()

In [ ]:
mask = (df['Severity'] == 'Severe') & (df["Type"] == "Cold")
df_severe_cold = df[mask].copy()
#il copy va usato perche pandas ritorna una vista del dataseto originale se dopo modifichiamo df_severe potrebbe dare errore

df_severe_cold['Month'] = df_severe_cold['StartTime(UTC)'].dt.month
#trasformandolo in datetime di pandas ci possiamo accedere con dt e abbiamo le sue funzionalità

#Creiamo la heatmap

#Contiamo per ongi combinazione di mese-anno il numero di eventi severe (collassa tutte le righe Gennaio 2016 in una sola contando il numero di occorrenze)
heatmap = df_severe_cold.groupby(['Year','Month']).size().reset_index(name='Count')
#Quindi dopo qui abbiamo le righe singole che sono Gennaio 2016 / febbraio 2016 ... agosto 2016 e per ognuna di queste righe uniche abbiamo il numero di count 

#Creiamo una tabella pivot da passare alla heatmap
pivot_table = heatmap.pivot(index='Year', columns='Month', values='Count').fillna(0)

fig = px.imshow(pivot_table, 
                title="Stagionalità e Frequenza degli Eventi Estremi di Fereddo (< -23.7°C) (2016-2022)",
                labels=dict(x="Mese", y="Anno", color="N eventi Estremi"),
                x=['Gen', 'Feb', 'Mar', 'Apr', 'Mag', 'Giu', 'Lug', 'Ago', 'Set', 'Ott', 'Nov', 'Dic']
                );

fig.show()

In [ ]:
mask = (df['Severity'] == 'Severe') & (df["Type"] == "Storm")
df_severe_storm = df[mask].copy()
#il copy va usato perche pandas ritorna una vista del dataseto originale se dopo modifichiamo df_severe potrebbe dare errore

df_severe_storm['Month'] = df_severe_storm['StartTime(UTC)'].dt.month
#trasformandolo in datetime di pandas ci possiamo accedere con dt e abbiamo le sue funzionalità

#Creiamo la heatmap

#Contiamo per ongi combinazione di mese-anno il numero di eventi severe (collassa tutte le righe Gennaio 2016 in una sola contando il numero di occorrenze)
heatmap = df_severe_storm.groupby(['Year','Month']).size().reset_index(name='Count')
#Quindi dopo qui abbiamo le righe singole che sono Gennaio 2016 / febbraio 2016 ... agosto 2016 e per ognuna di queste righe uniche abbiamo il numero di count 

#Creiamo una tabella pivot da passare alla heatmap
pivot_table = heatmap.pivot(index='Year', columns='Month', values='Count').fillna(0)

fig = px.imshow(pivot_table, 
                title="Stagionalità e Frequenza degli Eventi Estremi di Tempeste (2016-2022)",
                labels=dict(x="Mese", y="Anno", color="N eventi Estremi"),
                x=['Gen', 'Feb', 'Mar', 'Apr', 'Mag', 'Giu', 'Lug', 'Ago', 'Set', 'Ott', 'Nov', 'Dic']
                );

fig.show()

## Percentuale di intensità per tipo di evento

Ci aspettiamo un trend di aumento degli eventi intensi, ma le aspettative vengono confermate solo per la nebbia, per gli altri eventi non emerge un trend significativo

In [ ]:
for e in df.Type.unique():
    df_type = df[df['Type'] == e].copy()
    
    # Conta eventi per Severity e Year
    df_count = df_type.groupby(['Year', 'Severity']).size().reset_index(name='Count')
    
    # Calcola percentuale per anno
    df_count['Pct'] = df_count.groupby('Year')['Count'].transform(lambda x: x / x.sum() * 100)
    
    severity_order = ['Light', 'Moderate', 'Heavy', 'Severe']
    
    fig = px.bar(
        df_count,
        x='Year',
        y='Pct',
        color='Severity',
        barmode='relative',  # stacked 100%
        title=f'Severity Distribution - {e}',
        labels={'Pct': '% Events'},
        category_orders={'Severity': severity_order}
    )
    fig.update_layout(
        xaxis=dict(tickmode='linear', dtick=1),
        yaxis=dict(ticksuffix='%'),
        legend=dict(traceorder='reversed')
    )
    fig.show()

## Precipitazione media in funzione della durata media nel tempo
Gli stati ad ovest 

In [ ]:
noaa_regions = {
    # Northeast
    'CT':'Northeast','DE':'Northeast','ME':'Northeast','MD':'Northeast',
    'MA':'Northeast','NH':'Northeast','NJ':'Northeast','NY':'Northeast',
    'PA':'Northeast','RI':'Northeast','VT':'Northeast',
    # East North Central
    'IA':'East North Central','MI':'East North Central',
    'MN':'East North Central','WI':'East North Central',
    # Central
    'IL':'Central','IN':'Central','KY':'Central','MO':'Central',
    'OH':'Central','TN':'Central','WV':'Central',
    # Southeast
    'AL':'Southeast','FL':'Southeast','GA':'Southeast',
    'NC':'Southeast','SC':'Southeast','VA':'Southeast',
    # West North Central
    'MT':'West North Central','NE':'West North Central',
    'ND':'West North Central','SD':'West North Central','WY':'West North Central',
    # South
    'AR':'South','KS':'South','LA':'South','MS':'South','OK':'South','TX':'South',
    # Southwest
    'AZ':'Southwest','CO':'Southwest','NM':'Southwest','UT':'Southwest',
    # Northwest
    'ID':'Northwest','OR':'Northwest','WA':'Northwest',
    # West
    'CA':'West','NV':'West',
}
# fonte: https://psl.noaa.gov/data/usclimdivs/data/

df['Region'] = df['State'].map(noaa_regions).fillna('Other')

In [ ]:
df_rain = df[df['Type'] == 'Rain'].copy()
df_rain['DurationHrs'] = (df_rain['EndTime(UTC)'] - df_rain['StartTime(UTC)']).dt.total_seconds() / 3600
df_rain = df_rain[(df_rain['DurationHrs'] > 0) & (df_rain['Precipitation(mm)'] > 0)]

cap_d = df_rain['DurationHrs'].quantile(0.99)
cap_p = df_rain['Precipitation(mm)'].quantile(0.99)
df_rain = df_rain[(df_rain['DurationHrs'] <= cap_d) & (df_rain['Precipitation(mm)'] <= cap_p)]

agg = df_rain.groupby(['State', 'Year', 'Region']).agg(
    AvgDuration=('DurationHrs', 'mean'),
    AvgPrecip=('Precipitation(mm)', 'mean'),
    Count=('EventId', 'count')
).reset_index()

fig = px.scatter(
    agg,
    x='AvgDuration',
    y='AvgPrecip',
    size='Count',
    color='Region',
    animation_frame='Year',
    hover_name='State',
    title='Eventi di pioggia per Stato: Durata vs Precipitazioni nel tempo',
    labels={
        'AvgDuration': 'Durata media [h]',
        'AvgPrecip': 'Precipitazione media [mm]',
        'Count': 'Numero Eventi'
    },
    size_max=50,
    template='plotly_dark'
)

fig.update_layout(
    xaxis=dict(range=[0, agg['AvgDuration'].quantile(0.98)]),
    yaxis=dict(range=[0, agg['AvgPrecip'].quantile(0.98)])
)
fig.show()

In [ ]:
df_snow = df[df['Type'] == 'Snow'].copy()
df_snow['DurationHrs'] = (df_snow['EndTime(UTC)'] - df_snow['StartTime(UTC)']).dt.total_seconds() / 3600
df_snow = df_snow[(df_snow['DurationHrs'] > 0) & (df_snow['Precipitation(mm)'] > 0)]

cap_d = df_snow['DurationHrs'].quantile(0.99)
cap_p = df_snow['Precipitation(mm)'].quantile(0.99)
df_snow = df_snow[(df_snow['DurationHrs'] <= cap_d) & (df_snow['Precipitation(mm)'] <= cap_p)]

agg = df_snow.groupby(['State', 'Year', 'Region']).agg(
    AvgDuration=('DurationHrs', 'mean'),
    AvgPrecip=('Precipitation(mm)', 'mean'),
    Count=('EventId', 'count')
).reset_index()

fig = px.scatter(
    agg,
    x='AvgDuration',
    y='AvgPrecip',
    size='Count',
    color='Region',
    animation_frame='Year',
    hover_name='State',
    title='Eventi di neve per Stato: Durata vs Precipitazioni nel tempo',
    labels={
        'AvgDuration': 'Durata media [h]',
        'AvgPrecip': 'Precipitazione media [mm]',
        'Count': 'Numero Eventi'
    },
    size_max=50,
    template='plotly_dark'
)

fig.update_layout(
    xaxis=dict(range=[0, agg['AvgDuration'].quantile(0.98)]),
    yaxis=dict(range=[0, agg['AvgPrecip'].quantile(0.98)])
)
fig.show()